In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded")

# %%
# Load processed data
print("Loading processed datasets...")

df = pd.read_csv('../data/processed/events_cleaned.csv', parse_dates=['event_time'])
hourly = pd.read_csv('../data/processed/hourly_aggregated.csv', parse_dates=['event_time'])
daily = pd.read_csv('../data/processed/daily_aggregated.csv', parse_dates=['date'])

print(f"✅ Main dataset: {df.shape}")
print(f"✅ Hourly data: {hourly.shape}")
print(f"✅ Daily data: {daily.shape}")



# %%
# Event type distribution
event_counts = df['event_type'].value_counts()
fig = px.pie(values=event_counts.values, names=event_counts.index,
             title='Event Type Distribution',
             color_discrete_sequence=['#00d4ff', '#ff00ff', '#ffff00'])
fig.update_layout(template='plotly_dark')
fig.show()

# %%
# Hourly pattern
hourly_pattern = df.groupby('hour').size()
fig = px.bar(x=hourly_pattern.index, y=hourly_pattern.values,
             title='Events by Hour of Day',
             labels={'x': 'Hour', 'y': 'Event Count'})
fig.update_traces(marker_color='#00d4ff')
fig.update_layout(template='plotly_dark', height=400)
fig.show()

# %%
# Day of week pattern
dow_pattern = df.groupby('day_name').size().reindex([
    'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'
])
fig = px.bar(x=dow_pattern.index, y=dow_pattern.values,
             title='Events by Day of Week',
             labels={'x': 'Day', 'y': 'Event Count'})
fig.update_traces(marker_color='#ff00ff')
fig.update_layout(template='plotly_dark', height=400)
fig.show()

✅ Libraries loaded
Loading processed datasets...
✅ Main dataset: (3821589, 23)
✅ Hourly data: (744, 6)
✅ Daily data: (31, 6)


In [2]:
# %% [markdown]
# ### 2. UNIVARIATE ANALYSIS

# %% [markdown]
# #### 2.1 Event Volume Over Time

# %%
# Daily event trend
fig = px.line(daily, x='date', y='total_events',
              title='Daily Event Volume Trend',
              labels={'total_events': 'Number of Events', 'date': 'Date'})

fig.update_traces(line_color='#00d4ff', line_width=2)
fig.update_layout(template='plotly_dark', height=400)
fig.show()

In [3]:
# Calculate correlation
corr = daily['unique_users'].corr(daily['total_events'])
print(f"Correlation between users and events: {corr:.3f}")

# %%
# Revenue vs events
fig = px.scatter(daily, x='total_events', y='total_revenue',
                 hover_data=['date'],
                 title='Events vs Revenue',
                 trendline='ols')
fig.update_traces(marker=dict(size=8, color='#ff00ff'))
fig.update_layout(template='plotly_dark', height=500)
fig.show()

Correlation between users and events: 0.954


In [4]:
# %% [markdown]
# #### 3.2 Temporal Correlations

# %%
# Correlation matrix
numeric_cols = ['unique_users', 'total_events', 'unique_products', 'total_revenue', 'unique_sessions']
corr_matrix = daily[numeric_cols].corr()

fig = px.imshow(corr_matrix, text_auto='.2f',
                title='Feature Correlation Matrix',
                color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1)
fig.update_layout(template='plotly_dark', height=500)
fig.show()

In [5]:
# %% [markdown]
# #### 3.3 Weekend vs Weekday

# %%
# Compare weekend vs weekday
weekend_comparison = df.groupby(['is_weekend', 'event_type']).size().reset_index(name='count')
weekend_comparison['day_type'] = weekend_comparison['is_weekend'].map({0: 'Weekday', 1: 'Weekend'})

fig = px.bar(weekend_comparison, x='event_type', y='count', color='day_type',
             barmode='group',
             title='Event Types: Weekend vs Weekday',
             labels={'count': 'Number of Events', 'event_type': 'Event Type'})
fig.update_layout(template='plotly_dark', height=400)
fig.show()

In [6]:
# %% [markdown]
# ### 4. MULTIVARIATE ANALYSIS

# %% [markdown]
# #### 4.1 Heatmap: Hour x Day of Week

# %%
# Create heatmap data
heatmap_data = df.groupby(['hour', 'day_of_week']).size().reset_index(name='count')
heatmap_pivot = heatmap_data.pivot(index='hour', columns='day_of_week', values='count')

# Rename columns
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
heatmap_pivot.columns = [day_names[int(col)] for col in heatmap_pivot.columns]

fig = go.Figure(data=go.Heatmap(
    z=heatmap_pivot.values,
    x=heatmap_pivot.columns,
    y=heatmap_pivot.index,
    colorscale='Plasma',
    hoverongaps=False
))

fig.update_layout(
    title='Event Heatmap: Hour vs Day of Week',
    xaxis_title='Day of Week',
    yaxis_title='Hour of Day',
    template='plotly_dark',
    height=600
)
fig.show()

In [7]:
# %% [markdown]
# #### 4.2 Top Brands Analysis

# %%
# Top 10 brands by events
top_brands = df[df['brand'] != 'Unknown']['brand'].value_counts().head(10)

fig = px.bar(x=top_brands.values, y=top_brands.index,
             orientation='h',
             title='Top 10 Brands by Event Count',
             labels={'x': 'Event Count', 'y': 'Brand'})
fig.update_traces(marker_color='#00d4ff')
fig.update_layout(template='plotly_dark', height=500)
fig.show()

# %%
# Brand performance: conversion rates
brand_conversion = df.groupby('brand').agg({
    'event_type': lambda x: (x == 'purchase').sum() / len(x) * 100
}).reset_index()
brand_conversion.columns = ['brand', 'conversion_rate']
brand_conversion = brand_conversion[brand_conversion['brand'] != 'Unknown'].nlargest(10, 'conversion_rate')

fig = px.bar(brand_conversion, x='brand', y='conversion_rate',
             title='Top 10 Brands by Conversion Rate',
             labels={'conversion_rate': 'Conversion Rate (%)', 'brand': 'Brand'})
fig.update_traces(marker_color='#ff00ff')
fig.update_layout(template='plotly_dark', height=500, xaxis_tickangle=-45)
fig.show()


In [8]:
# %% [markdown]
# #### 4.3 Category Analysis

# %%
# Top categories
top_categories = df[df['main_category'] != 'Uncategorized']['main_category'].value_counts().head(10)

fig = px.pie(values=top_categories.values, names=top_categories.index,
             title='Top 10 Product Categories',
             hole=0.3)
fig.update_layout(template='plotly_dark')
fig.show()

In [9]:
# %% [markdown]
# ### 5. TEMPORAL PATTERNS

# %% [markdown]
# #### 5.1 Time of Day Analysis

# %%
# Events by time of day
time_of_day_counts = df['time_of_day'].value_counts().reindex(['Morning', 'Afternoon', 'Evening', 'Night'])

fig = px.bar(x=time_of_day_counts.index, y=time_of_day_counts.values,
             title='Events by Time of Day',
             labels={'x': 'Time of Day', 'y': 'Event Count'})
fig.update_traces(marker_color='#ffff00')
fig.update_layout(template='plotly_dark', height=400)
fig.show()

In [10]:
# %%
# Average session duration proxy (events per session)
events_per_session = df.groupby('user_session').size().describe()
print("Events per Session Statistics:")
print(events_per_session)

Events per Session Statistics:
count    851378.000000
mean          4.488058
std          12.335595
min           1.000000
25%           1.000000
50%           1.000000
75%           3.000000
max        1166.000000
dtype: float64


In [11]:
# %% [markdown]
# #### 5.2 User Behavior Patterns

# %%
# User activity distribution
user_activity = df.groupby('user_id').size()

fig = go.Figure()
fig.add_trace(go.Histogram(x=user_activity.values, nbinsx=50, marker_color='#00d4ff'))
fig.update_layout(
    title='User Activity Distribution',
    xaxis_title='Events per User',
    yaxis_title='Number of Users',
    template='plotly_dark',
    height=400
)
fig.show()

print(f"Median events per user: {user_activity.median():.0f}")
print(f"Mean events per user: {user_activity.mean():.2f}")

# %%
# Power users (top 1%)
threshold = user_activity.quantile(0.99)
power_users = user_activity[user_activity > threshold]
print(f"\nPower Users (top 1%):")
print(f"Threshold: {threshold:.0f} events")
print(f"Number of power users: {len(power_users):,}")
print(f"Total events by power users: {power_users.sum():,} ({power_users.sum()/len(df)*100:.1f}%)")

Median events per user: 2
Mean events per user: 9.81

Power Users (top 1%):
Threshold: 145 events
Number of power users: 3,875
Total events by power users: 1,046,498 (27.4%)


In [12]:
# %% [markdown]
# ### 6. ANOMALY DETECTION

# %%
# Detect anomalies in daily events using z-score
daily['events_zscore'] = np.abs(stats.zscore(daily['total_events']))
anomalies = daily[daily['events_zscore'] > 2.5]

print(f"Detected {len(anomalies)} anomalous days")
print("\nAnomaly dates:")
print(anomalies[['date', 'total_events', 'events_zscore']])

# %%
# Visualize anomalies
fig = go.Figure()

# Normal days
normal_days = daily[daily['events_zscore'] <= 2.5]
fig.add_trace(go.Scatter(
    x=normal_days['date'],
    y=normal_days['total_events'],
    mode='lines',
    name='Normal',
    line=dict(color='#00d4ff', width=2)
))

# Anomalies
fig.add_trace(go.Scatter(
    x=anomalies['date'],
    y=anomalies['total_events'],
    mode='markers',
    name='Anomalies',
    marker=dict(size=15, color='red', symbol='x', line=dict(width=2, color='white'))
))

fig.update_layout(
    title='Daily Events with Anomaly Detection',
    xaxis_title='Date',
    yaxis_title='Total Events',
    template='plotly_dark',
    height=500
)
fig.show()

Detected 1 anomalous days

Anomaly dates:
        date  total_events  events_zscore
1 2019-10-02        188101       3.174623


In [13]:
# %% [markdown]
# ### 7. CONVERSION FUNNEL ANALYSIS

# %%
# Calculate funnel metrics
total_views = len(df[df['event_type'] == 'view'])
total_cart = len(df[df['event_type'] == 'cart'])
total_purchase = len(df[df['event_type'] == 'purchase'])

view_to_cart = (total_cart / total_views) * 100 if total_views > 0 else 0
cart_to_purchase = (total_purchase / total_cart) * 100 if total_cart > 0 else 0
view_to_purchase = (total_purchase / total_views) * 100 if total_views > 0 else 0

print("Conversion Funnel Metrics:")
print("=" * 50)
print(f"Total Views: {total_views:,}")
print(f"Total Cart Additions: {total_cart:,}")
print(f"Total Purchases: {total_purchase:,}")
print(f"\nView → Cart: {view_to_cart:.2f}%")
print(f"Cart → Purchase: {cart_to_purchase:.2f}%")
print(f"View → Purchase: {view_to_purchase:.2f}%")

Conversion Funnel Metrics:
Total Views: 1,815,733
Total Cart Additions: 1,190,259
Total Purchases: 242,842

View → Cart: 65.55%
Cart → Purchase: 20.40%
View → Purchase: 13.37%


In [14]:
# %%
# Funnel visualization
funnel_data = pd.DataFrame({
    'Stage': ['Views', 'Cart', 'Purchase'],
    'Count': [total_views, total_cart, total_purchase],
    'Percentage': [100, view_to_cart, view_to_purchase]
})

fig = go.Figure(go.Funnel(
    y=funnel_data['Stage'],
    x=funnel_data['Count'],
    textinfo="value+percent initial",
    marker=dict(color=['#00d4ff', '#ff00ff', '#ffff00'])
))

fig.update_layout(
    title='E-commerce Conversion Funnel',
    template='plotly_dark',
    height=500
)
fig.show()

In [15]:
# %% [markdown]
# ### 8. KEY OBSERVATIONS

# %%
print("=" * 70)
print("KEY OBSERVATIONS FROM EXPLORATORY ANALYSIS")
print("=" * 70)

observations = [
    "\n1. TEMPORAL PATTERNS:",
    f"   • Peak activity hours: {hourly_pattern.idxmax()}:00",
    f"   • Lowest activity hours: {hourly_pattern.idxmin()}:00",
    f"   • Most active day: {dow_pattern.idxmax()}",
    f"   • Least active day: {dow_pattern.idxmin()}",
    
    "\n2. USER BEHAVIOR:",
    f"   • Median events per user: {user_activity.median():.0f}",
    f"   • Top 1% users generate {power_users.sum()/len(df)*100:.1f}% of events",
    
    "\n3. CONVERSION METRICS:",
    f"   • View to Cart rate: {view_to_cart:.2f}%",
    f"   • Cart to Purchase rate: {cart_to_purchase:.2f}%",
    f"   • Overall conversion: {view_to_purchase:.2f}%",
    
    "\n4. ANOMALIES:",
    f"   • Detected {len(anomalies)} anomalous days",
    f"   • Largest spike: {anomalies['total_events'].max():,.0f} events on {anomalies.loc[anomalies['total_events'].idxmax(), 'date'].strftime('%Y-%m-%d')}",
    
    "\n5. CORRELATIONS:",
    f"   • Users vs Events correlation: {daily['unique_users'].corr(daily['total_events']):.3f}",
    f"   • Events vs Revenue correlation: {daily['total_events'].corr(daily['total_revenue']):.3f}"
]

for obs in observations:
    print(obs)

print("\n" + "=" * 70)

KEY OBSERVATIONS FROM EXPLORATORY ANALYSIS

1. TEMPORAL PATTERNS:
   • Peak activity hours: 19:00
   • Lowest activity hours: 1:00
   • Most active day: Wednesday
   • Least active day: Saturday

2. USER BEHAVIOR:
   • Median events per user: 2
   • Top 1% users generate 27.4% of events

3. CONVERSION METRICS:
   • View to Cart rate: 65.55%
   • Cart to Purchase rate: 20.40%
   • Overall conversion: 13.37%

4. ANOMALIES:
   • Detected 1 anomalous days
   • Largest spike: 188,101 events on 2019-10-02

5. CORRELATIONS:
   • Users vs Events correlation: 0.954
   • Events vs Revenue correlation: 0.993



In [16]:
# %% [markdown]
# ### 9. Data Summary for Next Steps

# %%
# Prepare summary for visualization notebook
summary = {
    'dataset_shape': df.shape,
    'time_range': f"{df['event_time'].min()} to {df['event_time'].max()}",
    'total_users': df['user_id'].nunique(),
    'total_products': df['product_id'].nunique(),
    'anomaly_dates': anomalies['date'].tolist(),
    'peak_hour': int(hourly_pattern.idxmax()),
    'conversion_rate': view_to_purchase,
    'top_category': top_categories.index[0],
    'top_brand': top_brands.index[0]
}

print("Summary Statistics for Chronosphere:")
print("=" * 50)
for key, value in summary.items():
    print(f"{key}: {value}")

# %%
print("\n✅ Exploratory analysis complete!")
print("📊 Key patterns identified")
print("🔍 Anomalies detected")
print("📈 Ready for visualization experiments!")

Summary Statistics for Chronosphere:
dataset_shape: (3821589, 23)
time_range: 2019-10-01 00:00:00+00:00 to 2019-10-31 23:59:54+00:00
total_users: 389483
total_products: 41431
anomaly_dates: [Timestamp('2019-10-02 00:00:00')]
peak_hour: 19
conversion_rate: 13.374323207211633
top_category: appliances
top_brand: runail

✅ Exploratory analysis complete!
📊 Key patterns identified
🔍 Anomalies detected
📈 Ready for visualization experiments!
